# Notebook 2: Model Training

This notebook demonstrates the training process for eye disease classification models.

## Models:
- Baseline CNN (from scratch)
- ResNet50 (Transfer Learning)
- EfficientNetB0 (Transfer Learning)

In [ ]:
# Imports
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras

# Add parent directory to path
sys.path.append('..')

from src.config import get_config, CLASS_NAMES
from src.models import create_model, compile_model
from src.data_loader import create_image_data_generators
from src.train import train_model, create_callbacks

%matplotlib inline

# Check GPU availability
print(f"TensorFlow version: {tf.__version__}")
print(f"GPU Available: {tf.config.list_physical_devices('GPU')}")

## 1. Load Configuration and Data

In [ ]:
# Load configuration
config = get_config()

data_dir = config.get('dataset.processed_data_path', '../data/processed')
classes = config.get('dataset.classes', CLASS_NAMES)
image_size = config.get('dataset.image_size', 224)

print(f"Data directory: {data_dir}")
print(f"Classes: {classes}")
print(f"Image size: {image_size}")

# Check if data exists
if not os.path.exists(data_dir):
    print(f"\n❌ Error: Processed data not found at {data_dir}")
    print("Please run preprocessing first:")
    print("  python main.py preprocess")

## 2. Create Data Generators

In [ ]:
# Create data generators
train_gen, val_gen, _ = create_image_data_generators(
    data_dir=data_dir,
    classes=classes,
    image_size=(image_size, image_size),
    batch_size=32,
    validation_split=0.15,
    test_split=0.15
)

print(f"Training samples: {train_gen.samples}")
print(f"Validation samples: {val_gen.samples}")
print(f"Batch size: {train_gen.batch_size}")

## 3. Model 1: Baseline CNN

In [ ]:
# Create baseline CNN model
cnn_model = create_model(
    model_name='cnn_baseline',
    input_shape=(image_size, image_size, 3),
    num_classes=len(classes),
    dropout_rate=0.5
)

# Compile model
cnn_model = compile_model(
    cnn_model,
    learning_rate=0.001,
    optimizer_name='adam'
)

# Print model summary
cnn_model.summary()

In [ ]:
# Train baseline CNN (reduced epochs for demonstration)
print("\nTraining Baseline CNN...")

callbacks = create_callbacks(
    model_name='cnn_baseline',
    save_dir='../models',
    logs_dir='../logs',
    config={
        'early_stopping': True,
        'early_stopping_patience': 5,
        'reduce_lr_on_plateau': True,
        'model_checkpoint': True
    }
)

history_cnn = cnn_model.fit(
    train_gen,
    epochs=10,  # Use 50 for full training
    validation_data=val_gen,
    callbacks=callbacks,
    verbose=1
)

## 4. Model 2: ResNet50 (Transfer Learning)

In [ ]:
# Create ResNet50 model
resnet_model = create_model(
    model_name='resnet50',
    input_shape=(image_size, image_size, 3),
    num_classes=len(classes),
    freeze_base=True,  # Freeze base layers initially
    dropout_rate=0.5
)

# Compile model
resnet_model = compile_model(
    resnet_model,
    learning_rate=0.0001,  # Lower LR for transfer learning
    optimizer_name='adam'
)

print(f"Total parameters: {resnet_model.count_params():,}")
print(f"Trainable parameters: {sum([w.size for w in resnet_model.trainable_weights]):,}")

In [ ]:
# Train ResNet50
print("\nTraining ResNet50...")

callbacks = create_callbacks(
    model_name='resnet50',
    save_dir='../models',
    logs_dir='../logs'
)

history_resnet = resnet_model.fit(
    train_gen,
    epochs=10,  # Use 50 for full training
    validation_data=val_gen,
    callbacks=callbacks,
    verbose=1
)

## 5. Model 3: EfficientNetB0

In [ ]:
# Create EfficientNet model
effnet_model = create_model(
    model_name='efficientnet',
    input_shape=(image_size, image_size, 3),
    num_classes=len(classes),
    freeze_base=True,
    dropout_rate=0.3
)

# Compile model
effnet_model = compile_model(
    effnet_model,
    learning_rate=0.0001,
    optimizer_name='adam'
)

print(f"Total parameters: {effnet_model.count_params():,}")
print(f"Trainable parameters: {sum([w.size for w in effnet_model.trainable_weights]):,}")

## 6. Compare Models

In [ ]:
# Plot training history comparison
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Accuracy
if 'history_cnn' in locals():
    axes[0].plot(history_cnn.history['val_accuracy'], label='CNN Baseline')
if 'history_resnet' in locals():
    axes[0].plot(history_resnet.history['val_accuracy'], label='ResNet50')
if 'history_effnet' in locals():
    axes[0].plot(history_effnet.history['val_accuracy'], label='EfficientNet')

axes[0].set_title('Validation Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Loss
if 'history_cnn' in locals():
    axes[1].plot(history_cnn.history['val_loss'], label='CNN Baseline')
if 'history_resnet' in locals():
    axes[1].plot(history_resnet.history['val_loss'], label='ResNet50')
if 'history_effnet' in locals():
    axes[1].plot(history_effnet.history['val_loss'], label='EfficientNet')

axes[1].set_title('Validation Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Save Final Models

Models are automatically saved during training via callbacks.
Check the `models/` directory for saved models.

In [ ]:
# List saved models
import glob

model_dir = '../models'
saved_models = glob.glob(os.path.join(model_dir, '**/*.h5'), recursive=True)

print("Saved Models:")
for model_path in saved_models:
    print(f"  - {model_path}")

## Next Steps

1. Evaluate models: See notebook 03_evaluation.ipynb
2. Make predictions: Use `python main.py predict`
3. Run web app: `python main.py app`